In [3]:
%pip install soundfile

Note: you may need to restart the kernel to use updated packages.


In [9]:
# ============================================
# CELL 1 — Environment + Groq
# ============================================

import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY was not found. "
        "Make sure the project .env file contains GROQ_API_KEY."
    )

groq_client = Groq(
    api_key=api_key
)

print("Groq client ready!")

Environment file: C:\Users\dmane\Downloads\Cogniv AI\.env
Environment loaded: True
Groq key available: True
HF_TOKEN available: True
VOICE INPUT AGENT — CONFIGURATION
Project: C:\Users\dmane\Cogniv AI
Indic ASR model: C:\Users\dmane\Downloads\Cogniv AI\models\indic-transcribe-core
Model exists: True
Groq key available: True
Microphone device: 1
Sample rate: 16000
Channels: 1
Frame: 20 ms


In [ ]:
# ============================================
# CELL 2 — Audio Dependencies
# ============================================

import sys
import sounddevice as sd
import numpy as np
import wave

from pywebrtc_audio import AudioProcessor

print("Python:", sys.version.split()[0])
print("sounddevice:", sd.__version__)
print("numpy:", np.__version__)
print("pywebrtc-audio: ready")

In [ ]:
# ============================================
# CELL 3 — Microphone / Audio Configuration
# ============================================

import sounddevice as sd

# --------------------------------------------
# Recording configuration
# --------------------------------------------

DEVICE = 1
SAMPLE_RATE = 16000
CHANNELS = 1

# --------------------------------------------
# Show available input devices
# --------------------------------------------

devices = sd.query_devices()

print("Available input devices:")
print("==============================")

for i, device in enumerate(devices):

    if device["max_input_channels"] > 0:

        print(
            f"[{i}] {device['name']}"
        )

        print(
            f"    Input channels: "
            f"{device['max_input_channels']}"
        )

        print(
            f"    Default sample rate: "
            f"{device['default_samplerate']}"
        )

        print()

# --------------------------------------------
# Default device
# --------------------------------------------

DEFAULT_INPUT_DEVICE = sd.default.device[0]

print("==============================")
print(
    "Default input device index:",
    DEFAULT_INPUT_DEVICE
)

# --------------------------------------------
# Selected device
# --------------------------------------------

print(
    "\nSelected microphone device:",
    DEVICE
)

print(
    "Selected microphone:"
)

print(
    sd.query_devices(
        DEVICE,
        kind="input"
    )
)

# --------------------------------------------
# Verify recording configuration
# --------------------------------------------

sd.check_input_settings(
    device=DEVICE,
    channels=CHANNELS,
    dtype="int16",
    samplerate=SAMPLE_RATE
)

print("\n==============================")
print("Microphone configuration is supported.")
print("==============================")

In [ ]:
# ============================================
# CELL 4 — Voice / VAD Configuration
# ============================================

# --------------------------------------------
# Frame configuration
# --------------------------------------------

FRAME_MS = 20

FRAME_SAMPLES = int(
    SAMPLE_RATE * FRAME_MS / 1000
)

# --------------------------------------------
# Voice activity detection
# --------------------------------------------

START_THRESHOLD = 0.60
END_THRESHOLD = 0.35

START_DURATION_MS = 100
SILENCE_DURATION_MS = 800

# --------------------------------------------
# Pre-roll
# --------------------------------------------

PRE_ROLL_MS = 300

# --------------------------------------------
# Safety timeout
# --------------------------------------------

MAX_DURATION_SECONDS = 10

# --------------------------------------------
# Output
# --------------------------------------------

OUTPUT_FILE = "live_utterance.wav"

# --------------------------------------------
# Derived values
# --------------------------------------------

START_FRAMES = max(
    1,
    int(START_DURATION_MS / FRAME_MS)
)

SILENCE_FRAMES = max(
    1,
    int(SILENCE_DURATION_MS / FRAME_MS)
)

PRE_ROLL_FRAMES = max(
    1,
    int(PRE_ROLL_MS / FRAME_MS)
)

print("==============================")
print("VOICE CONFIGURATION")
print("==============================")

print("Sample rate:", SAMPLE_RATE)
print("Channels:", CHANNELS)
print("Device:", DEVICE)

print("Frame:", FRAME_MS, "ms")
print("Frame samples:", FRAME_SAMPLES)

print("Start threshold:", START_THRESHOLD)
print("End threshold:", END_THRESHOLD)

print("Start frames required:", START_FRAMES)
print("Silence frames required:", SILENCE_FRAMES)

print("Pre-roll:", PRE_ROLL_MS, "ms")
print("Maximum duration:", MAX_DURATION_SECONDS, "seconds")

print("Output file:", OUTPUT_FILE)

In [ ]:
# ============================================
# CELL 5 — Basic Microphone Recording Test
# ============================================

TEST_FILE = "mic_test.wav"
TEST_DURATION = 5

print("==============================")
print("MICROPHONE TEST")
print("==============================")
print()
print("Recording for 5 seconds...")
print("Please say:")
print()
print("    I drink water at 10:30")
print()
print("Speak naturally.")
print()

audio = sd.rec(
    int(TEST_DURATION * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=CHANNELS,
    dtype="float32",
    device=DEVICE
)

sd.wait()

# Convert float32 → 16-bit PCM
audio_int16 = np.int16(
    np.clip(audio, -1, 1) * 32767
)

# Save WAV
with wave.open(
    TEST_FILE,
    "wb"
) as wav_file:

    wav_file.setnchannels(
        CHANNELS
    )

    wav_file.setsampwidth(
        2
    )

    wav_file.setframerate(
        SAMPLE_RATE
    )

    wav_file.writeframes(
        audio_int16.tobytes()
    )

print()
print("==============================")
print("Recording completed")
print("==============================")
print("File:", TEST_FILE)
print("Duration:", TEST_DURATION, "seconds")
print("Sample rate:", SAMPLE_RATE)
print("Channels:", CHANNELS)
print("Format: 16-bit PCM")

In [ ]:
# ============================================
# CELL 6 — Local Audio Preprocessing
# ============================================

INPUT_FILE = "mic_test.wav"
CLEAN_FILE = "mic_clean.wav"

# --------------------------------------------
# Read recorded WAV
# --------------------------------------------

with wave.open(
    INPUT_FILE,
    "rb"
) as wav_file:

    sample_rate = wav_file.getframerate()
    channels = wav_file.getnchannels()
    sample_width = wav_file.getsampwidth()
    frame_count = wav_file.getnframes()

    raw_audio = np.frombuffer(
        wav_file.readframes(frame_count),
        dtype=np.int16
    )

# --------------------------------------------
# Validate audio format
# --------------------------------------------

if sample_rate != SAMPLE_RATE:

    raise ValueError(
        f"Expected {SAMPLE_RATE} Hz audio, "
        f"got {sample_rate} Hz."
    )

if channels != CHANNELS:

    raise ValueError(
        f"Expected {CHANNELS} channel, "
        f"got {channels} channels."
    )

if sample_width != 2:

    raise ValueError(
        "Expected 16-bit PCM audio."
    )

# --------------------------------------------
# Create WebRTC audio processor
# --------------------------------------------

processor = AudioProcessor(
    sample_rate=SAMPLE_RATE,
    num_channels=CHANNELS,

    noise_suppression=True,
    high_pass_filter=True,
    auto_gain_control=True,

    # 0 = weakest
    # 3 = strongest
    ns_level=2
)

# --------------------------------------------
# Process audio
# --------------------------------------------

clean_audio = processor.process(
    raw_audio
)

clean_audio = np.asarray(
    clean_audio,
    dtype=np.int16
)

# --------------------------------------------
# Save processed audio
# --------------------------------------------

with wave.open(
    CLEAN_FILE,
    "wb"
) as wav_file:

    wav_file.setnchannels(
        CHANNELS
    )

    wav_file.setsampwidth(
        2
    )

    wav_file.setframerate(
        SAMPLE_RATE
    )

    wav_file.writeframes(
        clean_audio.tobytes()
    )

# --------------------------------------------
# Diagnostics
# --------------------------------------------

print("==============================")
print("AUDIO PREPROCESSING COMPLETE")
print("==============================")

print("Input:", INPUT_FILE)
print("Output:", CLEAN_FILE)

print("Sample rate:", SAMPLE_RATE)
print("Channels:", CHANNELS)
print("Format: 16-bit PCM")

print(
    "Speech probability:",
    round(
        float(
            processor.speech_probability
        ),
        3
    )
)

try:

    print(
        "Current gain:",
        round(
            float(
                processor.gain_db
            ),
            2
        ),
        "dB"
    )

except Exception:

    pass

In [ ]:
# ============================================
# CELL 7 — Whisper Test on Clean Audio
# ============================================

CLEAN_FILE = "mic_clean.wav"

print("==============================")
print("WHISPER — CLEAN AUDIO TEST")
print("==============================")
print("File:", CLEAN_FILE)
print("Language: auto")
print()

with open(
    CLEAN_FILE,
    "rb"
) as audio_file:

    transcription = groq_client.audio.transcriptions.create(
        file=audio_file,
        model="whisper-large-v3-turbo",
        response_format="verbose_json",
        temperature=0
    )

print("==============================")
print("TRANSCRIPTION RESULT")
print("==============================")

print(
    "Text:",
    transcription.text
)

print(
    "Detected language:",
    getattr(
        transcription,
        "language",
        None
    )
)

In [ ]:
# ============================================
# CELL 8 — Whisper Quality Diagnostics
# ============================================

import json

CLEAN_FILE = "mic_clean.wav"

with open(
    CLEAN_FILE,
    "rb"
) as audio_file:

    transcription = groq_client.audio.transcriptions.create(
        file=audio_file,
        model="whisper-large-v3-turbo",
        response_format="verbose_json",
        timestamp_granularities=["segment"],
        temperature=0
    )

print("==============================")
print("WHISPER QUALITY DIAGNOSTICS")
print("==============================")

print("Text:")
print(transcription.text)

print()

print("Detected language:")
print(
    getattr(
        transcription,
        "language",
        None
    )
)

print()
print("==============================")
print("RAW RESPONSE")
print("==============================")

print(
    json.dumps(
        transcription.model_dump()
        if hasattr(transcription, "model_dump")
        else transcription,
        indent=2,
        default=str
    )
)

In [ ]:
# ============================================
# CELL 9 — RAW vs CLEAN Whisper Comparison
# ============================================

def transcribe_verbose(audio_file):

    with open(
        audio_file,
        "rb"
    ) as file:

        result = groq_client.audio.transcriptions.create(
            file=file,
            model="whisper-large-v3-turbo",
            response_format="verbose_json",
            timestamp_granularities=["segment"],
            temperature=0
        )

    return result


def print_result(label, result):

    print()
    print(label)
    print("------------------------------")

    print(
        "Text:",
        result.text
    )

    print(
        "Language:",
        getattr(
            result,
            "language",
            None
        )
    )

    segments = result.segments

    if not segments:

        print(
            "No segment metadata."
        )

        return

    segment = segments[0]

    print(
        "Avg logprob:",
        segment.get(
            "avg_logprob"
        )
    )

    print(
        "No-speech probability:",
        segment.get(
            "no_speech_prob"
        )
    )

    print(
        "Compression ratio:",
        segment.get(
            "compression_ratio"
        )
    )

    print(
        "Start:",
        segment.get(
            "start"
        )
    )

    print(
        "End:",
        segment.get(
            "end"
        )
    )


# --------------------------------------------
# TRANSCRIBE BOTH FILES
# --------------------------------------------

raw_result = transcribe_verbose(
    "mic_test.wav"
)

clean_result = transcribe_verbose(
    "mic_clean.wav"
)


# --------------------------------------------
# DISPLAY RESULTS
# --------------------------------------------

print("==============================")
print("RAW vs CLEAN COMPARISON")
print("==============================")

print_result(
    "RAW AUDIO",
    raw_result
)

print_result(
    "CLEAN AUDIO",
    clean_result
)

print()
print("==============================")
print("COMPARISON COMPLETE")
print("==============================")

In [ ]:
# ============================================
# CELL 10 — Live Voice Endpoint Test
# ============================================

from collections import deque
from pywebrtc_audio import AudioProcessor

print("==============================")
print("LIVE VOICE ENDPOINT TEST")
print("==============================")
print()
print("Speak naturally.")
print("The recording will stop automatically")
print("after you stop speaking.")
print()

processor = AudioProcessor(
    sample_rate=SAMPLE_RATE,
    num_channels=CHANNELS,
    noise_suppression=True,
    high_pass_filter=True,
    auto_gain_control=True,
    ns_level=2
)

pre_roll = deque(
    maxlen=PRE_ROLL_FRAMES
)

recorded_frames = []

speech_started = False
silence_frames = 0

stream = sd.InputStream(
    samplerate=SAMPLE_RATE,
    channels=CHANNELS,
    dtype="int16",
    blocksize=FRAME_SAMPLES,
    device=DEVICE
)

stream.start()

try:

    max_frames = int(
        MAX_DURATION_SECONDS
        * 1000
        / FRAME_MS
    )

    for frame_index in range(
        max_frames
    ):

        frame, overflowed = stream.read(
            FRAME_SAMPLES
        )

        frame = np.asarray(
            frame,
            dtype=np.int16
        ).reshape(-1)

        clean_frame = processor.process(
            frame
        )

        clean_frame = np.asarray(
            clean_frame,
            dtype=np.int16
        )

        probability = float(
            processor.speech_probability
        )

        pre_roll.append(
            clean_frame.copy()
        )

        if not speech_started:

            if probability >= START_THRESHOLD:

                speech_started = True

                print(
                    f"Speech started "
                    f"(VAD={probability:.2f})"
                )

                recorded_frames.extend(
                    list(pre_roll)
                )

        else:

            recorded_frames.append(
                clean_frame.copy()
            )

            if probability <= END_THRESHOLD:

                silence_frames += 1

            else:

                silence_frames = 0

            if silence_frames >= SILENCE_FRAMES:

                print(
                    f"Speech ended "
                    f"(VAD={probability:.2f})"
                )

                break

finally:

    stream.stop()
    stream.close()


print()

if not speech_started:

    print("==============================")
    print("NO SPEECH DETECTED")
    print("==============================")

else:

    audio = np.concatenate(
        recorded_frames
    )

    with wave.open(
        OUTPUT_FILE,
        "wb"
    ) as wav_file:

        wav_file.setnchannels(
            CHANNELS
        )

        wav_file.setsampwidth(
            2
        )

        wav_file.setframerate(
            SAMPLE_RATE
        )

        wav_file.writeframes(
            audio.tobytes()
        )

    duration = (
        len(audio) / SAMPLE_RATE
    )

    print("==============================")
    print("UTTERANCE CAPTURED")
    print("==============================")

    print(
        "File:",
        OUTPUT_FILE
    )

    print(
        "Duration:",
        round(duration, 2),
        "seconds"
    )

    print(
        "Samples:",
        len(audio)
    )

In [ ]:
# ============================================
# CELL 11 — Live Voice → Whisper
# ============================================

print("==============================")
print("LIVE VOICE → WHISPER TEST")
print("==============================")
print()
print("Speak naturally.")
print("The recording will stop automatically.")
print()

# --------------------------------------------
# CAPTURE
# --------------------------------------------

processor = AudioProcessor(
    sample_rate=SAMPLE_RATE,
    num_channels=CHANNELS,
    noise_suppression=True,
    high_pass_filter=True,
    auto_gain_control=True,
    ns_level=2
)

pre_roll = deque(
    maxlen=PRE_ROLL_FRAMES
)

recorded_frames = []

speech_started = False
silence_frames = 0

stream = sd.InputStream(
    samplerate=SAMPLE_RATE,
    channels=CHANNELS,
    dtype="int16",
    blocksize=FRAME_SAMPLES,
    device=DEVICE
)

stream.start()

try:

    max_frames = int(
        MAX_DURATION_SECONDS
        * 1000
        / FRAME_MS
    )

    for frame_index in range(
        max_frames
    ):

        frame, overflowed = stream.read(
            FRAME_SAMPLES
        )

        frame = np.asarray(
            frame,
            dtype=np.int16
        ).reshape(-1)

        clean_frame = processor.process(
            frame
        )

        clean_frame = np.asarray(
            clean_frame,
            dtype=np.int16
        )

        probability = float(
            processor.speech_probability
        )

        pre_roll.append(
            clean_frame.copy()
        )

        # ----------------------------
        # WAIT FOR SPEECH
        # ----------------------------

        if not speech_started:

            if probability >= START_THRESHOLD:

                speech_started = True

                print(
                    f"Speech started "
                    f"(VAD={probability:.2f})"
                )

                recorded_frames.extend(
                    list(pre_roll)
                )

        # ----------------------------
        # RECORD SPEECH
        # ----------------------------

        else:

            recorded_frames.append(
                clean_frame.copy()
            )

            if probability <= END_THRESHOLD:

                silence_frames += 1

            else:

                silence_frames = 0

            if silence_frames >= SILENCE_FRAMES:

                print(
                    f"Speech ended "
                    f"(VAD={probability:.2f})"
                )

                break

finally:

    stream.stop()
    stream.close()


# --------------------------------------------
# SAVE CAPTURED AUDIO
# --------------------------------------------

if not speech_started:

    raise RuntimeError(
        "No speech detected."
    )

audio = np.concatenate(
    recorded_frames
)

with wave.open(
    OUTPUT_FILE,
    "wb"
) as wav_file:

    wav_file.setnchannels(
        CHANNELS
    )

    wav_file.setsampwidth(
        2
    )

    wav_file.setframerate(
        SAMPLE_RATE
    )

    wav_file.writeframes(
        audio.tobytes()
    )


duration = len(audio) / SAMPLE_RATE


# --------------------------------------------
# WHISPER
# --------------------------------------------

print()
print("==============================")
print("TRANSCRIBING...")
print("==============================")

with open(
    OUTPUT_FILE,
    "rb"
) as audio_file:

    result = groq_client.audio.transcriptions.create(
        file=audio_file,
        model="whisper-large-v3-turbo",
        response_format="verbose_json",
        temperature=0
    )


# --------------------------------------------
# DISPLAY
# --------------------------------------------

print()
print("==============================")
print("VOICE RESULT")
print("==============================")

print(
    "Text:",
    result.text
)

print(
    "Detected language:",
    getattr(
        result,
        "language",
        None
    )
)

print(
    "Duration:",
    round(
        duration,
        2
    ),
    "seconds"
)

print(
    "Audio file:",
    OUTPUT_FILE
)

print()
print("==============================")
print("LIVE VOICE TEST COMPLETE")
print("==============================")

In [ ]:
# CELL 12 — Install Indic speech recognition dependencies

%pip install -U transformers torch torchaudio

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import HfApi

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN was not found in the .env file."
    )

api = HfApi(token=hf_token)

try:
    info = api.model_info(
        "bodhan-ai/indic-transcribe-core"
    )

    print("==============================")
    print("HUGGING FACE ACCESS VERIFIED")
    print("==============================")
    print("Model:", info.id)
    print("Gated access: available")
    print("Token detected: YES")
    print("==============================")

except Exception as e:
    print("==============================")
    print("HUGGING FACE ACCESS FAILED")
    print("==============================")
    print(type(e).__name__)
    print(str(e))

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN was not found in the .env file."
    )

MODEL_ID = "bodhan-ai/indic-transcribe-core"

print("==============================")
print("DOWNLOADING INDIC TRANSCRIBE")
print("==============================")
print("Model:", MODEL_ID)
print("This may take some time...")

model_dir = snapshot_download(
    repo_id=MODEL_ID,
    token=hf_token
)

print("==============================")
print("MODEL DOWNLOAD COMPLETE")
print("==============================")
print("Model directory:")
print(model_dir)
print("==============================")

In [ ]:
import os
import wave
import numpy as np
import sounddevice as sd

from pywebrtc_audio import AudioProcessor


class VoiceInputAgent:

    def __init__(
        self,
        client,
        model="whisper-large-v3-turbo",
        device=1,
        sample_rate=16000,
        frame_ms=20,
        start_threshold=0.60,
        end_threshold=0.35,
        start_duration_ms=100,
        silence_duration_ms=800,
        pre_roll_ms=300,
        max_duration=10
    ):
        self.client = client
        self.model = model

        self.device = device
        self.sample_rate = sample_rate
        self.frame_ms = frame_ms
        self.frame_samples = int(
            sample_rate * frame_ms / 1000
        )

        self.start_threshold = start_threshold
        self.end_threshold = end_threshold

        self.start_frames_required = max(
            1,
            int(start_duration_ms / frame_ms)
        )

        self.silence_frames_required = max(
            1,
            int(silence_duration_ms / frame_ms)
        )

        self.pre_roll_frames = max(
            1,
            int(pre_roll_ms / frame_ms)
        )

        self.max_duration = max_duration

        self.audio_processor = AudioProcessor(
            sample_rate=sample_rate,
            num_channels=1,
            noise_suppression=True,
            high_pass_filter=True,
            auto_gain_control=True,
            ns_level=2
        )

        self.indian_languages = {
            "as": "Assamese",
            "bn": "Bengali",
            "gu": "Gujarati",
            "hi": "Hindi",
            "kn": "Kannada",
            "ml": "Malayalam",
            "mr": "Marathi",
            "ne": "Nepali",
            "or": "Odia",
            "pa": "Punjabi",
            "sa": "Sanskrit",
            "ta": "Tamil",
            "te": "Telugu",
            "ur": "Urdu",
            "en": "English"
        }

    def _process_frame(self, frame):
        """
        Apply local microphone preprocessing.
        """

        frame = np.asarray(
            frame,
            dtype=np.int16
        ).reshape(-1)

        processed = self.audio_processor.process(
            frame
        )

        return np.asarray(
            processed,
            dtype=np.int16
        ).reshape(-1)

    def _speech_probability(self):
        """
        Return the speech probability from the
        WebRTC audio processor.
        """

        probability = getattr(
            self.audio_processor,
            "speech_probability",
            0.0
        )

        return float(probability)

    def capture_utterance(
        self,
        output_file="/mnt/data/live_utterance.wav"
    ):
        """
        Listen to the physical microphone and automatically
        capture one spoken utterance.
        """

        print("\n==============================")
        print("Listening...")
        print("==============================")
        print("Speak naturally.")
        print("Recording will stop after you finish speaking.\n")

        pre_roll = []

        utterance_frames = []

        speech_started = False

        speech_start_count = 0
        silence_count = 0

        max_frames = int(
            self.max_duration * 1000 / self.frame_ms
        )

        frame_counter = 0

        with sd.InputStream(
            samplerate=self.sample_rate,
            channels=1,
            dtype="int16",
            blocksize=self.frame_samples,
            device=self.device
        ) as stream:

            while frame_counter < max_frames:

                data, overflowed = stream.read(
                    self.frame_samples
                )

                frame = data[:, 0]

                processed = self._process_frame(
                    frame
                )

                probability = self._speech_probability()

                # Maintain pre-roll buffer.
                pre_roll.append(processed.copy())

                if len(pre_roll) > self.pre_roll_frames:
                    pre_roll.pop(0)

                # ----------------------------------
                # BEFORE SPEECH
                # ----------------------------------

                if not speech_started:

                    if probability >= self.start_threshold:

                        speech_start_count += 1

                    else:

                        speech_start_count = 0

                    if (
                        speech_start_count
                        >= self.start_frames_required
                    ):

                        speech_started = True

                        print(
                            f"Speech started "
                            f"(VAD={probability:.2f})"
                        )

                        # Include a little audio before
                        # the detected speech onset.
                        utterance_frames.extend(
                            pre_roll
                        )

                        silence_count = 0

                # ----------------------------------
                # DURING SPEECH
                # ----------------------------------

                else:

                    utterance_frames.append(
                        processed.copy()
                    )

                    if probability <= self.end_threshold:

                        silence_count += 1

                    else:

                        silence_count = 0

                    if (
                        silence_count
                        >= self.silence_frames_required
                    ):

                        print(
                            f"Speech ended "
                            f"(VAD={probability:.2f})"
                        )

                        break

                frame_counter += 1

        if not utterance_frames:

            raise RuntimeError(
                "No speech was detected."
            )

        audio = np.concatenate(
            utterance_frames
        )

        # Save WAV.
        with wave.open(
            output_file,
            "wb"
        ) as wav_file:

            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(
                self.sample_rate
            )

            wav_file.writeframes(
                audio.astype(
                    np.int16
                ).tobytes()
            )

        duration = len(audio) / self.sample_rate

        print("\n==============================")
        print("Utterance captured successfully")
        print("==============================")
        print(f"File: {output_file}")
        print(f"Duration: {duration:.2f} seconds")

        return {
            "audio_path": output_file,
            "duration": duration,
            "sample_rate": self.sample_rate
        }

    def transcribe_audio(
        self,
        audio_path,
        language="auto"
    ):
        """
        Transcribe an audio file.

        language="auto":
            Let Whisper detect the language.

        language="te", "hi", "bn", etc.:
            Explicitly specify the language.
        """

        if not os.path.exists(audio_path):

            raise FileNotFoundError(
                f"Audio file not found: {audio_path}"
            )

        request = {
            "model": self.model,
            "response_format": "verbose_json",
            "temperature": 0
        }

        if language != "auto":
            request["language"] = language

        with open(
            audio_path,
            "rb"
        ) as audio_file:

            response = (
                self.client
                .audio
                .transcriptions
                .create(
                    file=audio_file,
                    **request
                )
            )

        text = response.text.strip()

        detected_language = getattr(
            response,
            "language",
            None
        )

        return {
            "text": text,
            "language": detected_language,
            "language_name": self.indian_languages.get(
                detected_language,
                "Unknown"
            ),
            "model": self.model,
            "source": "microphone"
        }

    def listen_and_transcribe(
        self,
        language="auto",
        output_file="/mnt/data/live_utterance.wav"
    ):
        """
        Complete voice pipeline:

        microphone
            ↓
        local preprocessing
            ↓
        VAD
            ↓
        utterance capture
            ↓
        Whisper
            ↓
        structured result
        """

        capture_result = self.capture_utterance(
            output_file=output_file
        )

        transcription = self.transcribe_audio(
            audio_path=capture_result["audio_path"],
            language=language
        )

        return {
            **transcription,
            "audio_path": capture_result["audio_path"],
            "duration": capture_result["duration"]
        }


# Recreate the agent with the new implementation.

voice_input_agent = VoiceInputAgent(
    client=groq_client,
    model="whisper-large-v3-turbo",
    device=1
)

print("VoiceInputAgent updated successfully!")

In [ ]:
audio_path = "WhatsApp Audio 2026-09-12 at 10.11.50.mpeg"

transcription = voice_input_agent.transcribe_audio(
    audio_path=audio_path,
    language="en"
)

print("Transcription:")
print(transcription)

In [ ]:
import sys

!{sys.executable} -m pip install sounddevice

In [ ]:
import sounddevice as sd

print("sounddevice version:", sd.__version__)
print("Audio device system ready!")

In [ ]:
import sounddevice as sd

print("Default input device:")
print(sd.query_devices(kind="input"))

print("\n\nAll input devices:")
print(sd.query_devices(kind="input"))

In [ ]:
import sounddevice as sd

devices = sd.query_devices()

print("Available input devices:\n")

for i, device in enumerate(devices):
    if device["max_input_channels"] > 0:
        print(
            f"[{i}] {device['name']}"
            f" | channels={device['max_input_channels']}"
            f" | sample_rate={device['default_samplerate']}"
        )

print("\nDefault input:")
print(sd.query_devices(kind="input"))

In [ ]:
import sounddevice as sd
import numpy as np
import wave

SAMPLE_RATE = 16000
CHANNELS = 1
DURATION = 5
OUTPUT_FILE = "mic_test.wav"

print("Recording for 5 seconds...")
print("Please say:")
print("  I drink water at 10:30")
print()

audio = sd.rec(
    int(DURATION * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=CHANNELS,
    dtype="float32",
    device=1
)

sd.wait()

print("Recording finished.")

# Convert float32 [-1, 1] → int16 WAV
audio_int16 = np.int16(
    np.clip(audio, -1, 1) * 32767
)

with wave.open(OUTPUT_FILE, "wb") as wav_file:
    wav_file.setnchannels(CHANNELS)
    wav_file.setsampwidth(2)
    wav_file.setframerate(SAMPLE_RATE)
    wav_file.writeframes(audio_int16.tobytes())

print(f"Saved: {OUTPUT_FILE}")

In [ ]:
import sys

!{sys.executable} -m pip install pywebrtc-audio

In [ ]:
import wave
import numpy as np
from pywebrtc_audio import AudioProcessor


INPUT_FILE = "mic_test.wav"
OUTPUT_FILE = "mic_clean.wav"


# ---------------------------------------------------------
# Read the raw WAV file
# ---------------------------------------------------------

with wave.open(INPUT_FILE, "rb") as wav_file:

    channels = wav_file.getnchannels()
    sample_width = wav_file.getsampwidth()
    sample_rate = wav_file.getframerate()
    frame_count = wav_file.getnframes()

    if sample_width != 2:
        raise ValueError(
            "This example expects 16-bit PCM WAV audio."
        )

    raw_audio = np.frombuffer(
        wav_file.readframes(frame_count),
        dtype=np.int16
    )


print("Input audio:")
print("Sample rate:", sample_rate)
print("Channels:", channels)
print("Frames:", frame_count)


# ---------------------------------------------------------
# Create local WebRTC audio processor
#
# Echo cancellation is OFF because we do not have a
# speaker/reference audio signal for this offline test.
# ---------------------------------------------------------

processor = AudioProcessor(
    sample_rate=sample_rate,
    num_channels=channels,
    noise_suppression=True,
    high_pass_filter=True,
    auto_gain_control=True,
    ns_level=2
)


# ---------------------------------------------------------
# Clean the audio
# ---------------------------------------------------------

clean_audio = processor.process(raw_audio)


print("\nAudio processing complete.")
print(
    "Final speech probability:",
    round(processor.speech_probability, 3)
)

try:
    print(
        "Final gain:",
        round(processor.gain_db, 2),
        "dB"
    )
except RuntimeError:
    pass


# ---------------------------------------------------------
# Save cleaned audio
# ---------------------------------------------------------

with wave.open(OUTPUT_FILE, "wb") as output_wav:

    output_wav.setnchannels(channels)
    output_wav.setsampwidth(2)
    output_wav.setframerate(sample_rate)

    output_wav.writeframes(
        clean_audio.astype(np.int16).tobytes()
    )


print(f"\nClean audio saved as: {OUTPUT_FILE}")

In [ ]:
clean_audio_path ="mic_clean.wav"

clean_transcription = voice_input_agent.transcribe_audio(
    audio_path=clean_audio_path,
    language="en"
)

print("Cleaned audio transcription:")
print(clean_transcription)

In [ ]:
import wave
import numpy as np
from pywebrtc_audio import AudioProcessor

audio_path = "mic_clean.wav"

with wave.open(audio_path, "rb") as wav_file:
    sample_rate = wav_file.getframerate()
    channels = wav_file.getnchannels()

    audio = np.frombuffer(
        wav_file.readframes(wav_file.getnframes()),
        dtype=np.int16
    )

processor = AudioProcessor(
    sample_rate=sample_rate,
    num_channels=channels,
    noise_suppression=True,
    high_pass_filter=True,
    auto_gain_control=True,
    ns_level=2
)

# Process the complete recording
clean_audio = processor.process(audio)

print("Speech probability:", round(processor.speech_probability, 3))
print("Current gain:", round(processor.gain_db, 2), "dB")

In [ ]:
import wave
import numpy as np
from pywebrtc_audio import AudioProcessor

audio_path ="mic_clean.wav"

with wave.open(audio_path, "rb") as wav_file:
    sample_rate = wav_file.getframerate()
    channels = wav_file.getnchannels()

    audio = np.frombuffer(
        wav_file.readframes(wav_file.getnframes()),
        dtype=np.int16
    )

processor = AudioProcessor(
    sample_rate=sample_rate,
    num_channels=channels,
    noise_suppression=True,
    high_pass_filter=True,
    auto_gain_control=True,
    ns_level=2
)

frame_size = int(sample_rate * 0.02)  # 20 ms
probabilities = []

for start in range(0, len(audio), frame_size):

    frame = audio[start:start + frame_size]

    if len(frame) < frame_size:
        break

    processor.process(frame)

    probabilities.append(
        processor.speech_probability
    )

probabilities = np.array(probabilities)

print("Number of frames:", len(probabilities))
print("Minimum speech probability:", round(float(probabilities.min()), 3))
print("Maximum speech probability:", round(float(probabilities.max()), 3))
print("Average speech probability:", round(float(probabilities.mean()), 3))

print("\nFrames classified as speech (> 0.5):",
      int(np.sum(probabilities > 0.5)))

print("Total frames:",
      len(probabilities))

In [ ]:
import wave
import numpy as np
from pywebrtc_audio import AudioProcessor

audio_path = "mic_clean.wav"

with wave.open(audio_path, "rb") as wav_file:
    sample_rate = wav_file.getframerate()
    channels = wav_file.getnchannels()

    audio = np.frombuffer(
        wav_file.readframes(wav_file.getnframes()),
        dtype=np.int16
    )

processor = AudioProcessor(
    sample_rate=sample_rate,
    num_channels=channels,
    noise_suppression=True,
    high_pass_filter=True,
    auto_gain_control=True,
    ns_level=2
)

# 20 ms frames
frame_size = int(sample_rate * 0.02)

# Endpointing parameters
START_THRESHOLD = 0.60
END_THRESHOLD = 0.35

START_FRAMES = 3       # 60 ms sustained speech
END_FRAMES = 25        # 500 ms sustained silence
PRE_ROLL_FRAMES = 10   # 200 ms before speech onset

speech_probs = []

for start in range(0, len(audio), frame_size):

    frame = audio[start:start + frame_size]

    if len(frame) < frame_size:
        break

    processor.process(frame)

    speech_probs.append(
        processor.speech_probability
    )

# -----------------------------------------
# Simulate speech endpoint detection
# -----------------------------------------

pre_roll = []
utterance_frames = []

speaking = False
speech_count = 0
silence_count = 0

start_frame = None
end_frame = None

for i, probability in enumerate(speech_probs):

    # Keep recent frames for pre-roll
    pre_roll.append(i)

    if len(pre_roll) > PRE_ROLL_FRAMES:
        pre_roll.pop(0)

    if not speaking:

        if probability >= START_THRESHOLD:
            speech_count += 1
        else:
            speech_count = 0

        if speech_count >= START_FRAMES:

            speaking = True

            start_frame = pre_roll[0]

            utterance_frames = list(pre_roll)

            silence_count = 0

    else:

        utterance_frames.append(i)

        if probability < END_THRESHOLD:
            silence_count += 1
        else:
            silence_count = 0

        if silence_count >= END_FRAMES:

            end_frame = i - END_FRAMES + 1

            speaking = False

            break

# -----------------------------------------
# Results
# -----------------------------------------

print("Total frames:", len(speech_probs))

if start_frame is None:

    print("No speech segment detected.")

else:

    if end_frame is None:
        end_frame = len(speech_probs) - 1

    start_time = start_frame * 0.02
    end_time = (end_frame + 1) * 0.02

    duration = end_time - start_time

    print("\nSpeech segment detected!")
    print("Start frame:", start_frame)
    print("End frame:", end_frame)

    print("Start time:", round(start_time, 2), "seconds")
    print("End time:", round(end_time, 2), "seconds")
    print("Detected speech duration:", round(duration, 2), "seconds")

In [ ]:
import wave
import numpy as np
from pywebrtc_audio import AudioProcessor

audio_path = "mic_clean.wav"

with wave.open(audio_path, "rb") as wav_file:
    sample_rate = wav_file.getframerate()
    channels = wav_file.getnchannels()

    audio = np.frombuffer(
        wav_file.readframes(wav_file.getnframes()),
        dtype=np.int16
    )

processor = AudioProcessor(
    sample_rate=sample_rate,
    num_channels=channels,
    noise_suppression=True,
    high_pass_filter=True,
    auto_gain_control=True,
    ns_level=2
)

frame_size = int(sample_rate * 0.02)

results = []

for i, start in enumerate(range(0, len(audio), frame_size)):

    frame = audio[start:start + frame_size]

    if len(frame) < frame_size:
        break

    processor.process(frame)

    results.append({
        "frame": i,
        "time": i * 0.02,
        "probability": processor.speech_probability
    })

print("Probability every 10 frames:\n")

for i in range(0, len(results), 10):

    chunk = results[i:i + 10]

    values = [x["probability"] for x in chunk]

    print(
        f"{chunk[0]['time']:4.2f}s - "
        f"{chunk[-1]['time'] + 0.02:4.2f}s : "
        f"avg={np.mean(values):.3f}, "
        f"min={np.min(values):.3f}, "
        f"max={np.max(values):.3f}"
    )

In [ ]:
import wave
import numpy as np

audio_path = "mic_clean.wav"

with wave.open(audio_path, "rb") as wav_file:
    sample_rate = wav_file.getframerate()
    audio = np.frombuffer(
        wav_file.readframes(wav_file.getnframes()),
        dtype=np.int16
    )

frame_size = int(sample_rate * 0.02)

print("20 ms frame analysis:\n")

for start in range(0, len(audio), frame_size):

    frame = audio[start:start + frame_size]

    if len(frame) < frame_size:
        break

    rms = np.sqrt(
        np.mean(
            frame.astype(np.float32) ** 2
        )
    )

    dbfs = 20 * np.log10(
        max(rms, 1) / 32768
    )

    if start / sample_rate >= 3.0:

        print(
            f"{start/sample_rate:4.2f}s "
            f"RMS={rms:7.1f} "
            f"dBFS={dbfs:6.1f}"
        )

In [ ]:
import wave
import numpy as np
from pywebrtc_audio import AudioProcessor


class VoiceEndpointDetector:

    def __init__(
        self,
        sample_rate=16000,
        channels=1,
        start_threshold=0.60,
        end_threshold=0.35,
        start_duration_ms=100,
        silence_duration_ms=800,
        pre_roll_ms=300
    ):

        self.sample_rate = sample_rate
        self.channels = channels

        self.frame_ms = 20
        self.frame_size = int(
            sample_rate * self.frame_ms / 1000
        )

        self.start_threshold = start_threshold
        self.end_threshold = end_threshold

        self.start_frames = max(
            1,
            int(start_duration_ms / self.frame_ms)
        )

        self.silence_frames = max(
            1,
            int(silence_duration_ms / self.frame_ms)
        )

        self.pre_roll_frames = max(
            1,
            int(pre_roll_ms / self.frame_ms)
        )

        self.processor = AudioProcessor(
            sample_rate=sample_rate,
            num_channels=channels,
            noise_suppression=True,
            high_pass_filter=True,
            auto_gain_control=True,
            ns_level=2
        )

    def detect(self, audio):

        speech_probabilities = []

        for start in range(
            0,
            len(audio),
            self.frame_size
        ):

            frame = audio[
                start:start + self.frame_size
            ]

            if len(frame) < self.frame_size:
                break

            self.processor.process(frame)

            speech_probabilities.append(
                self.processor.speech_probability
            )

        # --------------------------------
        # Endpoint state machine
        # --------------------------------

        speaking = False

        speech_count = 0
        silence_count = 0

        pre_roll = []

        start_frame = None
        end_frame = None

        for i, probability in enumerate(
            speech_probabilities
        ):

            # Maintain rolling pre-roll buffer
            pre_roll.append(i)

            if len(pre_roll) > self.pre_roll_frames:
                pre_roll.pop(0)

            if not speaking:

                if probability >= self.start_threshold:
                    speech_count += 1
                else:
                    speech_count = 0

                if speech_count >= self.start_frames:

                    speaking = True

                    start_frame = pre_roll[0]

                    silence_count = 0

            else:

                if probability < self.end_threshold:
                    silence_count += 1
                else:
                    silence_count = 0

                if silence_count >= self.silence_frames:

                    end_frame = (
                        i - self.silence_frames + 1
                    )

                    break

        # --------------------------------
        # No speech detected
        # --------------------------------

        if start_frame is None:

            return {
                "detected": False,
                "start_time": None,
                "end_time": None,
                "duration": 0.0,
                "speech_probabilities":
                    speech_probabilities
            }

        # --------------------------------
        # Recording ended at file boundary
        # --------------------------------

        if end_frame is None:

            end_frame = (
                len(speech_probabilities) - 1
            )

        start_time = (
            start_frame * self.frame_ms / 1000
        )

        end_time = (
            (end_frame + 1)
            * self.frame_ms
            / 1000
        )

        return {
            "detected": True,
            "start_time": start_time,
            "end_time": end_time,
            "duration": end_time - start_time,
            "start_frame": start_frame,
            "end_frame": end_frame,
            "speech_probabilities":
                speech_probabilities
        }


# ----------------------------------------
# Load our cleaned microphone recording
# ----------------------------------------

audio_path = "mic_clean.wav"

with wave.open(audio_path, "rb") as wav_file:

    sample_rate = wav_file.getframerate()
    channels = wav_file.getnchannels()

    audio = np.frombuffer(
        wav_file.readframes(
            wav_file.getnframes()
        ),
        dtype=np.int16
    )


# ----------------------------------------
# Create detector
# ----------------------------------------

detector = VoiceEndpointDetector(
    sample_rate=sample_rate,
    channels=channels,
    start_threshold=0.60,
    end_threshold=0.35,
    start_duration_ms=100,
    silence_duration_ms=800,
    pre_roll_ms=300
)


# ----------------------------------------
# Detect utterance
# ----------------------------------------

result = detector.detect(audio)


print("Speech detected:", result["detected"])

if result["detected"]:

    print(
        "Start:",
        round(result["start_time"], 2),
        "seconds"
    )

    print(
        "End:",
        round(result["end_time"], 2),
        "seconds"
    )

    print(
        "Duration:",
        round(result["duration"], 2),
        "seconds"
    )

else:

    print("No speech detected.")

In [ ]:
import sounddevice as sd
import numpy as np
import wave
import time

from pywebrtc_audio import AudioProcessor


# ==========================================
# Configuration
# ==========================================

SAMPLE_RATE = 16000
CHANNELS = 1

FRAME_MS = 20
FRAME_SIZE = int(
    SAMPLE_RATE * FRAME_MS / 1000
)

START_THRESHOLD = 0.60
END_THRESHOLD = 0.35

START_FRAMES = 5        # 100 ms
END_FRAMES = 40         # 800 ms

PRE_ROLL_FRAMES = 15    # 300 ms

MAX_RECORDING_SECONDS = 10


# ==========================================
# WebRTC processor
# ==========================================

processor = AudioProcessor(
    sample_rate=SAMPLE_RATE,
    num_channels=CHANNELS,
    noise_suppression=True,
    high_pass_filter=True,
    auto_gain_control=True,
    ns_level=2
)


# ==========================================
# State
# ==========================================

pre_roll = []

recording = []

speaking = False

speech_count = 0
silence_count = 0

start_time = None


print("Listening...")
print("Speak naturally.")
print("The recording will stop automatically.")


# ==========================================
# Microphone callback
# ==========================================

def callback(indata, frames, time_info, status):

    global pre_roll
    global recording
    global speaking
    global speech_count
    global silence_count
    global start_time

    if status:
        print("Audio status:", status)

    # Convert microphone frame to int16
    audio = (
        indata[:, 0]
        * 32767
    ).astype(np.int16)

    # Process through WebRTC
    clean_audio = processor.process(audio)

    probability = processor.speech_probability

    # --------------------------------------
    # Pre-roll buffer
    # --------------------------------------

    pre_roll.append(
        clean_audio.copy()
    )

    if len(pre_roll) > PRE_ROLL_FRAMES:
        pre_roll.pop(0)


    # ======================================
    # Waiting for speech
    # ======================================

    if not speaking:

        if probability >= START_THRESHOLD:

            speech_count += 1

        else:

            speech_count = 0


        if speech_count >= START_FRAMES:

            speaking = True

            start_time = time.time()

            # Include pre-roll
            recording = [
                frame.copy()
                for frame in pre_roll
            ]

            print(
                "\nSpeech started "
                f"(VAD={probability:.2f})"
            )

            silence_count = 0


    # ======================================
    # Currently recording speech
    # ======================================

    else:

        recording.append(
            clean_audio.copy()
        )


        if probability < END_THRESHOLD:

            silence_count += 1

        else:

            silence_count = 0


        elapsed = (
            time.time() - start_time
        )


        # ----------------------------------
        # Speech ended
        # ----------------------------------

        if silence_count >= END_FRAMES:

            print(
                "\nSpeech ended "
                f"(VAD={probability:.2f})"
            )

            raise sd.CallbackStop


        # ----------------------------------
        # Safety timeout
        # ----------------------------------

        if elapsed >= MAX_RECORDING_SECONDS:

            print(
                "\nMaximum recording duration reached."
            )

            raise sd.CallbackStop


# ==========================================
# Start microphone
# ==========================================

with sd.InputStream(
    samplerate=SAMPLE_RATE,
    channels=CHANNELS,
    dtype="float32",
    blocksize=FRAME_SIZE,
    device=1,
    callback=callback
):

    while True:

        time.sleep(0.1)

        if not speaking:
            continue

        elapsed = (
            time.time() - start_time
        )

        if elapsed >= MAX_RECORDING_SECONDS:
            break

        # CallbackStop will terminate the stream
        # when the utterance ends.

In [ ]:
import sounddevice as sd
import numpy as np
import wave
import time

from pywebrtc_audio import AudioProcessor


# ==========================================
# Configuration
# ==========================================

SAMPLE_RATE = 16000
CHANNELS = 1

FRAME_MS = 20
FRAME_SIZE = int(
    SAMPLE_RATE * FRAME_MS / 1000
)

START_THRESHOLD = 0.60
END_THRESHOLD = 0.35

START_FRAMES = 5        # 100 ms
END_FRAMES = 40         # 800 ms
PRE_ROLL_FRAMES = 15    # 300 ms

MAX_RECORDING_SECONDS = 10

OUTPUT_FILE = "live_utterance.wav"


# ==========================================
# Processor
# ==========================================

processor = AudioProcessor(
    sample_rate=SAMPLE_RATE,
    num_channels=CHANNELS,
    noise_suppression=True,
    high_pass_filter=True,
    auto_gain_control=True,
    ns_level=2
)


# ==========================================
# State
# ==========================================

pre_roll = []
recording = []

speaking = False

speech_count = 0
silence_count = 0

start_time = None
finished = False


print("Listening...")
print("Speak naturally.")
print("Recording will stop after you finish speaking.")


# ==========================================
# Callback
# ==========================================

def callback(indata, frames, time_info, status):

    global pre_roll
    global recording
    global speaking
    global speech_count
    global silence_count
    global start_time
    global finished

    if status:
        print("Audio status:", status)

    # Convert float32 microphone audio → int16
    audio = (
        indata[:, 0] * 32767
    ).astype(np.int16)

    # Local preprocessing
    clean_audio = processor.process(audio)

    probability = processor.speech_probability

    # --------------------------------------
    # Pre-roll
    # --------------------------------------

    pre_roll.append(
        clean_audio.copy()
    )

    if len(pre_roll) > PRE_ROLL_FRAMES:
        pre_roll.pop(0)


    # ======================================
    # Waiting for speech
    # ======================================

    if not speaking:

        if probability >= START_THRESHOLD:

            speech_count += 1

        else:

            speech_count = 0


        if speech_count >= START_FRAMES:

            speaking = True

            start_time = time.time()

            recording = [
                frame.copy()
                for frame in pre_roll
            ]

            silence_count = 0

            print(
                f"\nSpeech started "
                f"(VAD={probability:.2f})"
            )


    # ======================================
    # Recording speech
    # ======================================

    else:

        recording.append(
            clean_audio.copy()
        )

        if probability < END_THRESHOLD:

            silence_count += 1

        else:

            silence_count = 0


        elapsed = (
            time.time() - start_time
        )


        # ----------------------------------
        # End of speech
        # ----------------------------------

        if silence_count >= END_FRAMES:

            print(
                f"\nSpeech ended "
                f"(VAD={probability:.2f})"
            )

            finished = True

            raise sd.CallbackStop


        # ----------------------------------
        # Safety limit
        # ----------------------------------

        if elapsed >= MAX_RECORDING_SECONDS:

            print(
                "\nMaximum recording duration reached."
            )

            finished = True

            raise sd.CallbackStop


# ==========================================
# Start microphone
# ==========================================

try:

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=CHANNELS,
        dtype="float32",
        blocksize=FRAME_SIZE,
        device=1,
        callback=callback
    ):

        while not finished:

            time.sleep(0.05)

except sd.CallbackStop:

    pass


# ==========================================
# Save detected utterance
# ==========================================

if len(recording) == 0:

    print("No utterance captured.")

else:

    captured_audio = np.concatenate(
        recording
    )

    with wave.open(
        OUTPUT_FILE,
        "wb"
    ) as wav_file:

        wav_file.setnchannels(CHANNELS)
        wav_file.setsampwidth(2)
        wav_file.setframerate(SAMPLE_RATE)

        wav_file.writeframes(
            captured_audio.astype(
                np.int16
            ).tobytes()
        )


    duration = (
        len(captured_audio)
        / SAMPLE_RATE
    )

    print("\n==============================")
    print("Utterance captured successfully")
    print("==============================")
    print("File:", OUTPUT_FILE)
    print("Duration:", round(duration, 2), "seconds")
    print("Samples:", len(captured_audio))

In [ ]:
# ==========================================
# Transcribe the automatically captured
# microphone utterance using Whisper
# ==========================================

audio_path = "live_utterance.wav"

text = voice_input_agent.transcribe_audio(
    audio_path=audio_path,
    language="en"
)

print("Whisper transcription:")
print(text)

In [ ]:
import os
from groq import Groq


class VoiceInputAgent:

    def __init__(
        self,
        client,
        model="whisper-large-v3-turbo"
    ):
        self.client = client
        self.model = model

        # Common Indian language codes supported by Whisper.
        self.indian_languages = {
            "as": "Assamese",
            "bn": "Bengali",
            "gu": "Gujarati",
            "hi": "Hindi",
            "kn": "Kannada",
            "ml": "Malayalam",
            "mr": "Marathi",
            "ne": "Nepali",
            "pa": "Punjabi",
            "sa": "Sanskrit",
            "ta": "Tamil",
            "te": "Telugu",
            "ur": "Urdu",
            "en": "English"
        }

    def transcribe_audio(
        self,
        audio_path,
        language="auto"
    ):
        """
        Transcribe patient speech.

        language="auto"
            Let Whisper detect the spoken language.

        language="te", "hi", "bn", etc.
            Explicitly specify the language when known.
        """

        if not os.path.exists(audio_path):
            raise FileNotFoundError(
                f"Audio file not found: {audio_path}"
            )

        # Build request.
        request = {
            "model": self.model,
            "response_format": "verbose_json",
            "temperature": 0
        }

        # Only send language when explicitly known.
        # If omitted, Whisper performs multilingual handling.
        if language != "auto":
            request["language"] = language

        with open(audio_path, "rb") as audio_file:

            response = self.client.audio.transcriptions.create(
                file=audio_file,
                **request
            )

        text = response.text.strip()

        detected_language = getattr(
            response,
            "language",
            None
        )

        return {
            "text": text,
            "language": detected_language,
            "language_name": self.indian_languages.get(
                detected_language,
                "Unknown"
            ),
            "model": self.model,
            "source": "microphone"
        }


voice_input_agent = VoiceInputAgent(
    client=groq_client
)

print("Multilingual VoiceInputAgent ready!")

In [ ]:
audio_path = "live_utterance.wav"

result = voice_input_agent.transcribe_audio(
    audio_path=audio_path,
    language="auto"
)

print("Voice result:")
print(result)

In [ ]:
# Check which voice-related functions currently exist

voice_functions = [
    name for name in globals()
    if any(word in name.lower() for word in [
        "capture",
        "utterance",
        "endpoint",
        "voice",
        "speech"
    ])
]

print("Voice-related objects currently in notebook:")
for name in sorted(voice_functions):
    print("-", name)

In [ ]:
import inspect

print("Constructor:")
print(inspect.signature(VoiceInputAgent))

print("\nMethods:")
for name in dir(VoiceInputAgent):
    if not name.startswith("_"):
        print("-", name)

In [ ]:
import os
import wave
import numpy as np
import sounddevice as sd

from pywebrtc_audio import AudioProcessor


class VoiceInputAgent:

    def __init__(
        self,
        client,
        model="whisper-large-v3-turbo",
        device=1,
        sample_rate=16000,
        frame_ms=20,
        start_threshold=0.60,
        end_threshold=0.35,
        start_duration_ms=100,
        silence_duration_ms=800,
        pre_roll_ms=300,
        max_duration=10
    ):
        self.client = client
        self.model = model
        self.device = device
        self.sample_rate = sample_rate

        self.frame_ms = frame_ms
        self.frame_samples = int(
            sample_rate * frame_ms / 1000
        )

        self.start_threshold = start_threshold
        self.end_threshold = end_threshold

        self.start_frames_required = max(
            1,
            int(start_duration_ms / frame_ms)
        )

        self.silence_frames_required = max(
            1,
            int(silence_duration_ms / frame_ms)
        )

        self.pre_roll_frames = max(
            1,
            int(pre_roll_ms / frame_ms)
        )

        self.max_duration = max_duration

        # Local audio preprocessing.
        self.audio_processor = AudioProcessor(
            sample_rate=sample_rate,
            num_channels=1,
            noise_suppression=True,
            high_pass_filter=True,
            auto_gain_control=True,
            ns_level=2
        )

        self.language_names = {
            "as": "Assamese",
            "bn": "Bengali",
            "en": "English",
            "gu": "Gujarati",
            "hi": "Hindi",
            "kn": "Kannada",
            "ml": "Malayalam",
            "mr": "Marathi",
            "ne": "Nepali",
            "or": "Odia",
            "pa": "Punjabi",
            "sa": "Sanskrit",
            "ta": "Tamil",
            "te": "Telugu",
            "ur": "Urdu"
        }

    def _process_frame(self, frame):

        frame = np.asarray(
            frame,
            dtype=np.int16
        ).reshape(-1)

        processed = self.audio_processor.process(
            frame
        )

        return np.asarray(
            processed,
            dtype=np.int16
        ).reshape(-1)

    def _get_speech_probability(self):

        probability = getattr(
            self.audio_processor,
            "speech_probability",
            0.0
        )

        return float(probability)

    def capture_utterance(
        self,
        output_file="live_utterance.wav"
    ):

        print("\n==============================")
        print("Listening...")
        print("==============================")
        print("Speak naturally.")
        print("Recording will stop automatically.\n")

        pre_roll = []
        utterance_frames = []

        speech_started = False

        speech_start_count = 0
        silence_count = 0

        max_frames = int(
            self.max_duration * 1000 / self.frame_ms
        )

        frame_counter = 0

        with sd.InputStream(
            samplerate=self.sample_rate,
            channels=1,
            dtype="int16",
            blocksize=self.frame_samples,
            device=self.device
        ) as stream:

            while frame_counter < max_frames:

                data, overflowed = stream.read(
                    self.frame_samples
                )

                frame = data[:, 0]

                processed = self._process_frame(
                    frame
                )

                probability = self._get_speech_probability()

                # Keep pre-roll audio.
                pre_roll.append(
                    processed.copy()
                )

                if len(pre_roll) > self.pre_roll_frames:
                    pre_roll.pop(0)

                # -----------------------------
                # WAITING FOR SPEECH
                # -----------------------------

                if not speech_started:

                    if probability >= self.start_threshold:
                        speech_start_count += 1
                    else:
                        speech_start_count = 0

                    if (
                        speech_start_count
                        >= self.start_frames_required
                    ):

                        speech_started = True

                        print(
                            f"Speech started "
                            f"(VAD={probability:.2f})"
                        )

                        utterance_frames.extend(
                            pre_roll
                        )

                        silence_count = 0

                # -----------------------------
                # SPEECH IN PROGRESS
                # -----------------------------

                else:

                    utterance_frames.append(
                        processed.copy()
                    )

                    if probability <= self.end_threshold:
                        silence_count += 1
                    else:
                        silence_count = 0

                    if (
                        silence_count
                        >= self.silence_frames_required
                    ):

                        print(
                            f"Speech ended "
                            f"(VAD={probability:.2f})"
                        )

                        break

                frame_counter += 1

        if not utterance_frames:

            raise RuntimeError(
                "No speech was detected."
            )

        audio = np.concatenate(
            utterance_frames
        )

        with wave.open(
            output_file,
            "wb"
        ) as wav_file:

            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(
                self.sample_rate
            )

            wav_file.writeframes(
                audio.astype(
                    np.int16
                ).tobytes()
            )

        duration = (
            len(audio) / self.sample_rate
        )

        print("\n==============================")
        print("Utterance captured successfully")
        print("==============================")
        print(f"File: {output_file}")
        print(f"Duration: {duration:.2f} seconds")

        return {
            "audio_path": output_file,
            "duration": duration
        }

    def transcribe_audio(
        self,
        audio_path,
        language="auto"
    ):

        if not os.path.exists(audio_path):

            raise FileNotFoundError(
                f"Audio file not found: {audio_path}"
            )

        request = {
            "model": self.model,
            "response_format": "verbose_json",
            "temperature": 0
        }

        # If language is explicitly known,
        # tell Whisper. Otherwise let it detect.
        if language != "auto":
            request["language"] = language

        with open(
            audio_path,
            "rb"
        ) as audio_file:

            response = (
                self.client
                .audio
                .transcriptions
                .create(
                    file=audio_file,
                    **request
                )
            )

        text = response.text.strip()

        detected_language = getattr(
            response,
            "language",
            None
        )

        # Normalize both possible response styles:
        # "te" / "Telugu"
        if detected_language:

            language_lower = str(
                detected_language
            ).lower()

            reverse_lookup = {
                name.lower(): code
                for code, name
                in self.language_names.items()
            }

            if language_lower in reverse_lookup:

                language_code = reverse_lookup[
                    language_lower
                ]

            elif language_lower in self.language_names:

                language_code = language_lower

            else:

                language_code = str(
                    detected_language
                )

        else:

            language_code = None

        language_name = self.language_names.get(
            language_code,
            str(detected_language)
            if detected_language
            else "Unknown"
        )

        return {
            "text": text,
            "language": language_code,
            "language_name": language_name,
            "model": self.model,
            "source": "microphone"
        }

    def listen_and_transcribe(
        self,
        language="auto",
        output_file="/mnt/data/live_utterance.wav"
    ):

        capture = self.capture_utterance(
            output_file=output_file
        )

        transcription = self.transcribe_audio(
            audio_path=capture["audio_path"],
            language=language
        )

        return {
            **transcription,
            "audio_path": capture["audio_path"],
            "duration": capture["duration"]
        }


# IMPORTANT:
# Recreate the object after redefining the class.

voice_input_agent = VoiceInputAgent(
    client=groq_client,
    model="whisper-large-v3-turbo",
    device=1
)

print("VoiceInputAgent recreated successfully.")
print()
print("listen_and_transcribe:",
      hasattr(
          voice_input_agent,
          "listen_and_transcribe"
      ))
print("capture_utterance:",
      hasattr(
          voice_input_agent,
          "capture_utterance"
      ))
print("transcribe_audio:",
      hasattr(
          voice_input_agent,
          "transcribe_audio"
      ))

In [ ]:
result = voice_input_agent.listen_and_transcribe(
    language="te",
    output_file="live_utterance.wav"
)

print("\n==============================")
print("TELUGU TEST")
print("==============================")
print("Text:", result["text"])
print("Language:", result["language"])
print("Language name:", result["language_name"])
print("Duration:", round(result["duration"], 2), "seconds")
print("Audio:", result["audio_path"])

In [ ]:
result = voice_input_agent.listen_and_transcribe(output_file="live_utterance.wav")

print("\n==============================")
print("AUTO LANGUAGE TEST")
print("==============================")
print("Text:", result["text"])
print("Language:", result["language"])
print("Language name:", result["language_name"])
print("Duration:", round(result["duration"], 2), "seconds")
print("Audio:", result["audio_path"])

In [ ]:
result_large = voice_input_agent.transcribe_audio(
    audio_path="live_utterance.wav",
    language="auto"
)

print("\n==============================")
print("LARGE V3 AUTO-DETECTION TEST")
print("==============================")
print("Text:", result_large["text"])
print("Language:", result_large["language"])
print("Language name:", result_large["language_name"])